# Adaptive Beamforming by Deep LEarning (ABLE)

This example demonstrates ...

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tue-bmd/zea/blob/main/docs/source/notebooks/models/custom_models_example.ipynb)
&nbsp;
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/tue-bmd/zea/blob/main/docs/source/notebooks/models/custom_models_example.ipynb)

In [1]:
%%capture
%pip install zea

In [2]:
import os

os.environ["KERAS_BACKEND"] = "jax"

In [3]:
import numpy as np
import keras
from IPython.display import display

import zea
from zea import load_file, init_device
from zea.visualize import set_mpl_style
from zea.ops import (
    Pipeline,
    PatchedGrid,
    EnvelopeDetect,
    Normalize,
    LogCompress,
    TOFCorrection,
    DelayAndSum,
    Lambda,
)

from zea.models.able import ABLE

zea: Using backend 'jax'


2025-10-23 13:04:36.308405: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761224676.436958    2123 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761224676.474094    2123 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761224676.750094    2123 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761224676.750164    2123 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761224676.750167    2123 computation_placer.cc:177] computation placer alr

Let's set the parameters for the beamforming grid.

In [4]:
grid_size_x = 100
grid_size_z = 100

In [5]:
device = init_device(verbose=False)
set_mpl_style()

## Load a Frame from the PICMUS Dataset

We use the zea loader to load a single frame of raw RF data from the PICMUS dataset, along with scan and probe parameters.

In [6]:
# Load a single frame of raw RF data from PICMUS
path = "hf://zeahub/picmus/database/experiments/contrast_speckle/contrast_speckle_expe_dataset_iq/contrast_speckle_expe_dataset_iq.hdf5"
data, scan, probe = load_file(
    path=path,
    indices=[0],
    data_type="raw_data",
)
zlims = (0, 0.06)
xlims = (-0.019, 0.019)
dynamic_range = (-50, 0)

scan.n_ch = data.shape[-1]  # iq data
scan.zlims = zlims
scan.xlims = xlims
scan.grid_size_x = grid_size_x
scan.grid_size_z = grid_size_z

zea: DEBUG Skipping invalid parameter 'n_frames'.


## Build a zea Pipeline for Beamforming and Image Formation

Let's build an example ultrasound image formation pipeline. We use a PatchedGrid pipeline for memory-efficient beamforming, followed by envelope detection, normalization, and log compression. Currently jit is not supported for backend set to "torch". If you really want to use jit (understandably so), you are best off using either "jax" or "tensorflow" as the backend.

In [9]:
pipeline = Pipeline(
    operations=[
        PatchedGrid(
            operations=[
                TOFCorrection(),
                Lambda(ABLE().call, name="ABLE Reconstruction"),
                DelayAndSum()
                ],
            num_patches=100,
            jit_options=None,
        ),
        EnvelopeDetect(),
        Normalize(),
        LogCompress(),
    ],
    with_batch_dim=True,
    jit_options=None,
)

We prepare the parameters for the pipeline and run it to obtain a B-mode image.

In [ ]:
parameters = pipeline.prepare_parameters(probe, scan, dynamic_range=dynamic_range)
parameters["demodulation_frequency"] = parameters["sampling_frequency"]

inputs = {pipeline.key: keras.ops.convert_to_tensor(data)}
outputs = pipeline(**inputs, **parameters)
bmode = outputs[pipeline.output_key]

bmode_img = zea.display.to_8bit(bmode[0], dynamic_range=dynamic_range)
display(bmode_img)